# Chương 3 – Dữ liệu cho Model Adaptation (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phuongtnd1994/ptnd-ModelAdaptationBook/blob/main/code/chapter03/chapter03_colab.ipynb)

1. **DatasetManifest** (§3.10) – chạy trên CPU, không cần key.
2. **Pipeline sinh dữ liệu tổng hợp 6 bước** (§3.7) – dùng **Claude làm teacher và judge** (API key của bạn, ~$0.05–0.5).
3. **Thí nghiệm chất lượng dữ liệu** (§3.1) – train 4 LoRA trên Financial PhraseBank (~30 phút, cần L4/A100).

## 0. Cài đặt môi trường

Chạy lần lượt các ô dưới đây **một lần** mỗi khi mở runtime mới. Ô đầu tiên clone repo, cài package của sách và tạo hàm `run()` để chạy lệnh và in log trực tiếp ra notebook.

> Nếu pip báo `ERROR: pip's dependency resolver ...` về `gradio`, `diffusers`, `huggingface-hub` hoặc `safetensors`: **bỏ qua được**. Đó là các thư viện Colab cài sẵn mà sách không dùng; sách cố ý giữ `transformers<5` và `safetensors<0.6`.

In [ ]:
import os, re, sys, subprocess

# Repo chứa code của sách. Nếu fork của bạn để private, đổi sang repo gốc:
#   https://github.com/bahree/ModelAdaptationBook.git
REPO_URL = "https://github.com/phuongtnd1994/ptnd-ModelAdaptationBook.git"
REPO_DIR = "/content/ModelAdaptationBook"
CODE_DIR = f"{REPO_DIR}/code"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.environ["PYTHONUNBUFFERED"] = "1"   # log của script con hiện ra ngay
os.environ["WANDB_DISABLED"] = "true"  # không dùng Weights & Biases trên Colab
# Sách chỉ dùng PyTorch. Không cho transformers nạp TensorFlow có sẵn trên Colab
# (tránh cảnh báo giả "Could not find cuda drivers" của TensorFlow).
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


def run(cmd: str) -> None:
    """Chạy một lệnh shell từ thư mục code/, in log trực tiếp, báo lỗi nếu thất bại."""
    # Một số script không tự tạo thư mục output -> tạo sẵn trước khi chạy.
    for flag, path in re.findall(r"--(output_dir|output|out)\s+[\"']?([^\s\"']+)", cmd):
        target = path if flag == "output_dir" or not os.path.splitext(path)[1] else os.path.dirname(path)
        if target:
            os.makedirs(os.path.join(CODE_DIR, target), exist_ok=True)
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {p.returncode}): {cmd}")


run('pip install -q -e ".[dev,chapter03]"')
print("\nĐã cài xong. Thư mục làm việc:", os.getcwd())

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | {mem:.1f} GB | compute capability {cap[0]}.{cap[1]}")
    if cap < (8, 0):
        print("CẢNH BÁO: GPU này (ví dụ T4) không hỗ trợ bf16 gốc. Các bước TRAINING "
              "trong sách dùng bf16 -> hãy đổi sang L4 hoặc A100 "
              "(Runtime > Change runtime type).")
else:
    print("Không có GPU. Chỉ các bước chạy trên CPU / gọi API hoạt động được.")

### Khoá API (Colab Secrets)

Mở biểu tượng 🔑 (Secrets) ở thanh bên trái, thêm:

- `ANTHROPIC_API_KEY` = API key Claude của bạn (**bắt buộc** cho notebook này)
- `HF_TOKEN` = token Hugging Face (không bắt buộc)

và bật *Notebook access*. Nếu chưa thêm, ô dưới sẽ hỏi bạn nhập tay.

In [ ]:
import os
from getpass import getpass


def get_secret(name, required=False):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    if required:
        return getpass(f"Nhập {name}: ")
    return None


os.environ["ANTHROPIC_API_KEY"] = get_secret("ANTHROPIC_API_KEY", required=True)
hf = get_secret("HF_TOKEN")  # không bắt buộc, giúp tải model nhanh hơn
if hf:
    os.environ["HF_TOKEN"] = hf
print("Đã nạp khoá:", [k for k in ("ANTHROPIC_API_KEY", "HF_TOKEN") if os.environ.get(k)])

## 1. DatasetManifest demo (CPU)

In [ ]:
run("python -m chapter03.ch03_datasetmanifest")

## 2. Pipeline sinh dữ liệu tổng hợp bằng Claude (6 bước)

Chạy từng bước ngay trong notebook. Trước tiên thay hàm `call_teacher` bằng phiên bản đọc đúng khối text của Claude và chừa token cho adaptive thinking. Đổi `TEACHER_MODEL` nếu muốn (ví dụ `claude-opus-5-5` — chất lượng cao hơn, đắt hơn).

In [ ]:
import anthropic
import chapter03.ch03_synthetic_data_generation as syn

TEACHER_MODEL = "claude-sonnet-5-5"   # model teacher mặc định của sách
_client = anthropic.Anthropic()


def call_teacher(prompt, max_tokens=4096, system=None):
    kwargs = {"system": system} if system else {}
    resp = _client.messages.create(
        model=TEACHER_MODEL,
        max_tokens=max(max_tokens, 4096),
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
        **kwargs,
    )
    return "".join(b.text for b in resp.content if b.type == "text")


syn.call_teacher = call_teacher   # các bước bên dưới sẽ gọi Claude qua hàm này

### Bước 1–2: Load seeds từ Financial PhraseBank và xem prompt sinh dữ liệu

In [ ]:
seeds = syn.load_seeds(n_per_category=20)
sample_prompt = syn.build_generation_prompt("positive", seeds, n_to_generate=5)
print(sample_prompt[:1500])

### Bước 3: Gọi Claude sinh ứng viên

`N_PER_CAT = 15` giữ chi phí thấp; tăng lên 50–100 cho bản production.

In [ ]:
N_PER_CAT = 15
candidates = syn.generate_candidates(seeds, n_per_cat=N_PER_CAT)
for ex in candidates[:5]:
    print(f"[{ex['label']:8s}] {ex['sentence'][:100]}")

### Bước 4: Quality gate (Claude làm judge)

In [ ]:
passed, failed, pass_rate = syn.run_quality_gate(candidates)
print(f"\nPass rate: {pass_rate:.0%}  |  passed={len(passed)}  failed={len(failed)}")
if failed:
    print("Ví dụ bị loại:", failed[0]["sentence"][:100], failed[0].get("scores"))

### Bước 5–6: Kiểm tra phân phối, trộn với dữ liệu thật và lưu kèm manifest

In [ ]:
alignment = syn.check_distribution_alignment(seeds, passed)
if passed:
    manifest = syn.mix_and_save(seeds, passed, output_dir="./ch03_synthetic_output")

In [ ]:
import json, pathlib
p = pathlib.Path("ch03_synthetic_output/manifest.json")
print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:4000] if p.exists() else f"Chưa có {p}")

## 3. Thí nghiệm chất lượng dữ liệu (§3.1)

Train cùng một model trên 4 phiên bản dữ liệu (AllAgree, 75Agree, 50Agree, Corrupted 20%) và so sánh trên cùng test set. **~25–35 phút trên L4/A100.**

In [ ]:
run("python -m chapter03.ch03_data_quality_explore")

In [ ]:
import json, pathlib
p = pathlib.Path("ch03_quality_experiment/results.json")
print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:4000] if p.exists() else f"Chưa có {p}")

In [ ]:
from IPython.display import Image, display
import pathlib
chart = pathlib.Path("ch03_quality_experiment/accuracy_chart.png")
display(Image(str(chart))) if chart.exists() else print("Chưa có biểu đồ")

## Lưu kết quả về Google Drive (tuỳ chọn)

Runtime Colab sẽ bị xoá khi ngắt kết nối. Chạy ô dưới để copy thư mục kết quả sang Drive.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/ModelAdaptationBook"
for p in ["ch03_synthetic_output", "ch03_quality_experiment"]:
    src = os.path.join(CODE_DIR, p)
    dst = os.path.join(DEST, p)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Đã lưu", p, "->", dst)
    elif os.path.isfile(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Đã lưu", p, "->", dst)